##### Copyright 2025 Google LLC.

In [1]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Handling Errors and Retries with the Gemini API

<a target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/quickstarts/Handling_Errors_and_Retries.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/></a>

When building applications with the Gemini API, requests can occasionally fail. Understanding *why* a request failed is crucial because it dictates how your code should react. 

For a full list of API errors, review the [Gemini API Troubleshooting Documentation](https://ai.google.dev/gemini-api/docs/troubleshooting).

## Error Types: When to Retry vs. When to Stop

Not all errors should trigger a retry. If your prompt contains invalid parameters, retrying the exact same request will result in the exact same error, wasting time and resources. 

**Errors you SHOULD retry (Transient Errors):**
*   **`429` (Resource Exhausted):** You have exceeded your rate limit or quota. *Action:* Pause and retry using exponential backoff.
*   **`500` (Internal Error) & `503` (Service Unavailable):** The model is temporarily overloaded or experiencing a glitch. *Action:* Pause and retry.
*   **`502` (Bad Gateway) & `504` (Gateway Timeout):** These are temporary network errors that occur between the client and the Google API frontend. Because these are usually caused by transient congestion or connection blips, they often resolve themselves in a matter of seconds. Action: Pause and retry.

**Errors you SHOULD NOT retry (Client Errors):**
*   **`400` (Bad Request):** Invalid parameters, unsupported model, or a prompt that is too long. *Action:* Stop. Fix the code or prompt before sending again.
*   **`403` (Forbidden):** Invalid API key or permission denied. *Action:* Stop. Check your credentials.
*   **`404` (Not Found):** The requested resource (e.g., a specific model name, a file in the File API, or a deleted cache) could not be found. Action: Stop. Verify the model name or resource ID.

<a name="setup"></a>
## Setup

### Install SDK

Install the SDK from [PyPI](https://github.com/googleapis/python-genai). It's recommended to always use the latest version.

In [ ]:
%pip install -U -q 'google-genai>=1.51.0'

### Setup your API key

To run the following cell, your API key must be stored in a Colab Secret named `GEMINI_API_KEY`. If you don't already have an API key, or you're not sure how to create a Colab Secret, see [Authentication](./Authentication.ipynb) for an example.

In [ ]:
from google.colab import userdata

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

### Choose a model

Select the model you want to use in this guide. You can either select one from the list or enter a model name manually. Keep in mind that some models, such as the 2.5 ones are thinking models and thus take slightly more time to respond. For more details, you can see [thinking notebook ![image](https://storage.googleapis.com/generativeai-downloads/images/colab_icon16.png)](./Get_started_thinking.ipynb) to learn how to control the thinking.

Feel free to select [Gemini 3 Pro](https://ai.google.dev/gemini-api/docs/models#gemini-3-pro) if you want to try our newest model, but keep in mind that it has no free tier.

For a full overview of all Gemini models, check the [documentation](https://ai.google.dev/gemini-api/docs/models/gemini).

In [ ]:
MODEL_ID = "gemini-3-flash-preview" # @param ["gemini-2.5-flash-lite", "gemini-3.1-flash-lite-preview", "gemini-2.5-flash", "gemini-3-flash-preview", "gemini-2.5-pro", "gemini-3.1-pro-preview"] {"allow-input":true, "isTemplate": true}

## Method 1: Use Built-in SDK HttpRetryOptions

For standard use cases, the `google-genai` SDK handles retries automatically. Configure the `HttpRetryOptions` when initializing the client. The SDK will automatically catch `429` and `5xx` errors and apply exponential backoff under the hood.

In [ ]:
from google import genai
from google.genai import types

client = genai.Client(
    api_key=GEMINI_API_KEY,
    http_options=types.HttpOptions(
        retry_options=types.HttpRetryOptions(
            attempts=5,                                  
            initial_delay=2.0,                           
            max_delay=60.0,                              
            http_status_codes=[429, 500, 502, 503, 504]  
        )
    )
)

response = client.models.generate_content(
    model=MODEL_ID,
    contents='Explain exponential backoff in one sentence.'
)
print(response.text)

## Method 2: Custom Error Handling and Backoff

For complex applications, you may want precise control over the retry logic. The following example demonstrates how to catch the `errors.APIError` exception, inspect the specific HTTP status code, and execute a custom exponential backoff loop based on the error type.

In [ ]:
from google import genai
from google.genai import errors
import time

# Re-initialize a standard client without automatic retries
client = genai.Client(api_key=GEMINI_API_KEY)

max_retries = 4
base_delay = 2.0

for attempt in range(max_retries):
    try:
        response = client.models.generate_content(
            model=MODEL_ID,
            contents='Write a short poem about resilience.'
        )
        print("Success!\n")
        print(response.text)
        break  # Break out of the loop if successful

    except errors.APIError as e:
        print(f"Attempt {attempt + 1} failed. Error Code: {e.code} - {e.message}")
        
        # Check if it is a transient error that can be retried
        if e.code in [429, 500, 502, 503, 504]:
            delay = base_delay * (2 ** attempt)  # Exponential backoff (2s, 4s, 8s...)
            print(f"Transient error detected. Retrying in {delay} seconds...\n")
            time.sleep(delay)
            
        # If it is a client error (e.g., 400 Bad Request), retrying won't help!
        elif e.code in [400, 403, 404]:
            print("Fatal client error detected. Please fix your request parameters. Stopping retries.")
            break
            
        else:
            print("Unexpected error encountered. Stopping retries.")
            break
else:
    print("Maximum retries reached. The request ultimately failed.")

## Next steps

<a name="next_steps"></a>

* [Prompting Strategies](./Prompting.ipynb): Learn how to craft better prompts to minimize errors and improve response quality.
* [Functional Calling](./Function_calling.ipynb): Teach the model how to use external tools and APIs when it encounters tasks it cannot solve alone.
* [Context Caching](./Caching.ipynb): Learn how to reduce costs and latency for large prompts by caching frequently used data.

Also check the other Gemini capabilities that you can find in the [Gemini quickstarts](https://github.com/google-gemini/cookbook/tree/main/quickstarts/).